# 02 — SQL Analysis
## MedIQ — Hospital Patient Analytics & Readmission Predictor
**Goal:** Load patient data into MySQL and write advanced SQL queries to answer real hospital operations questions — patient risk stratification, department analysis, medication patterns, and readmission drivers.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# MySQL connection
DB_USER     = "root"
DB_PASSWORD = "root"
DB_HOST     = "localhost"
DB_NAME     = "mediq_db"

# Create database
import mysql.connector
conn = mysql.connector.connect(
    host=DB_HOST, user=DB_USER, password=DB_PASSWORD
)
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
cursor.close()
conn.close()

# SQLAlchemy engine
engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
)

print("✅ Connected to MySQL successfully")
print(f"   Database: {DB_NAME}")

✅ Connected to MySQL successfully
   Database: mediq_db


In [2]:
# Load clean data
df = pd.read_csv('../data/processed/mediq_clean.csv')

# Select relevant columns for SQL analysis
sql_cols = [
    'race', 'gender', 'age', 'age_numeric',
    'admission_type_id', 'discharge_disposition_id',
    'time_in_hospital', 'medical_specialty',
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient',
    'number_diagnoses', 'max_glu_serum', 'A1Cresult',
    'insulin', 'change', 'diabetesMed',
    'readmitted', 'readmitted_binary',
    'diag_1_category', 'num_meds_changed',
    'num_meds_active', 'utilization_score'
]

df_sql = df[sql_cols].copy()

# Load into MySQL
df_sql.to_sql(
    name='patients',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM patients"))
    count = result.fetchone()[0]

print(f"✅ Data loaded into MySQL")
print(f"   Table : patients")
print(f"   Rows  : {count:,}")

✅ Data loaded into MySQL
   Table : patients
   Rows  : 101,745


In [3]:
def run_query(sql, title=""):
    with engine.connect() as conn:
        df_result = pd.read_sql(text(sql), conn)
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    print(df_result.to_string(index=False))
    return df_result

print("✅ Helper function ready")

✅ Helper function ready


In [4]:
q1 = """
SELECT 
    CASE 
        WHEN number_inpatient >= 5 THEN 'Very High Risk'
        WHEN number_inpatient >= 3 THEN 'High Risk'
        WHEN number_inpatient >= 1 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END                                          AS risk_tier,
    COUNT(*)                                     AS total_patients,
    SUM(readmitted_binary)                       AS readmissions,
    ROUND(AVG(readmitted_binary) * 100, 2)       AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)              AS avg_days_in_hospital,
    ROUND(AVG(num_medications), 2)               AS avg_medications
FROM patients
GROUP BY risk_tier
ORDER BY readmission_rate_pct DESC;
"""
df_q1 = run_query(q1, "Q1 — Patient Risk Stratification")


  Q1 — Patient Risk Stratification
     risk_tier  total_patients  readmissions  readmission_rate_pct  avg_days_in_hospital  avg_medications
Very High Risk            2016         734.0                 36.41                  4.89            17.56
     High Risk            5030        1073.0                 21.33                  5.01            17.51
   Medium Risk           27079        3841.0                 14.18                  4.71            16.57
      Low Risk           67620        5704.0                  8.44                  4.21            15.64


In [5]:
q2 = """
SELECT 
    medical_specialty,
    COUNT(*)                                     AS total_patients,
    ROUND(AVG(readmitted_binary) * 100, 2)       AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)              AS avg_stay_days,
    ROUND(AVG(num_medications), 2)               AS avg_medications,
    ROUND(AVG(num_lab_procedures), 2)            AS avg_lab_procedures
FROM patients
WHERE medical_specialty != 'Unknown'
GROUP BY medical_specialty
HAVING total_patients >= 100
ORDER BY readmission_rate_pct DESC
LIMIT 15;
"""
df_q2 = run_query(q2, "Q2 — Top 15 Departments by Readmission Rate")


  Q2 — Top 15 Departments by Readmission Rate
                medical_specialty  total_patients  readmission_rate_pct  avg_stay_days  avg_medications  avg_lab_procedures
              Hematology/Oncology             206                 19.42           4.83            15.15               32.13
                         Oncology             348                 18.97           5.14            16.18               41.30
PhysicalMedicineandRehabilitation             391                 15.35           8.87            13.60               38.11
                       Nephrology            1612                 15.32           5.02            17.77               43.26
                 Surgery-Vascular             533                 13.88           4.12            20.98               33.21
                       Psychiatry             854                 12.18           6.39            11.59               38.96
           Family/GeneralPractice            7440                 11.87           4.3

In [6]:
q3 = """
WITH medication_profile AS (
    SELECT 
        CASE 
            WHEN num_meds_changed = 0 THEN 'No Change'
            WHEN num_meds_changed = 1 THEN '1 Med Changed'
            WHEN num_meds_changed = 2 THEN '2 Meds Changed'
            ELSE '3+ Meds Changed'
        END                                      AS med_change_group,
        readmitted_binary,
        time_in_hospital,
        num_medications
    FROM patients
)
SELECT
    med_change_group,
    COUNT(*)                                     AS total_patients,
    SUM(readmitted_binary)                       AS readmissions,
    ROUND(AVG(readmitted_binary) * 100, 2)       AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)              AS avg_stay_days,
    ROUND(AVG(num_medications), 2)               AS avg_medications
FROM medication_profile
GROUP BY med_change_group
ORDER BY readmission_rate_pct DESC;
"""
df_q3 = run_query(q3, "Q3 — Medication Change Impact (CTE)")


  Q3 — Medication Change Impact (CTE)
med_change_group  total_patients  readmissions  readmission_rate_pct  avg_stay_days  avg_medications
 3+ Meds Changed             112          16.0                 14.29           7.01            23.57
   1 Med Changed           26254        3445.0                 13.12           5.07            18.90
  2 Meds Changed            1316         157.0                 11.93           6.43            21.85
       No Change           74063        7734.0                 10.44           4.12            14.89


In [7]:
q4 = """
WITH patient_segments AS (
    SELECT
        diag_1_category,
        age,
        ROUND(AVG(readmitted_binary) * 100, 2)   AS readmission_rate,
        COUNT(*)                                  AS patient_count,
        ROUND(AVG(time_in_hospital), 2)           AS avg_stay,
        ROUND(AVG(utilization_score), 2)          AS avg_utilization
    FROM patients
    GROUP BY diag_1_category, age
    HAVING patient_count >= 50
),
ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY diag_1_category
            ORDER BY readmission_rate DESC
        )                                         AS risk_rank
    FROM patient_segments
)
SELECT
    diag_1_category,
    age,
    patient_count,
    readmission_rate,
    avg_stay,
    avg_utilization,
    risk_rank
FROM ranked
WHERE risk_rank = 1
ORDER BY readmission_rate DESC;
"""
df_q4 = run_query(q4, "Q4 — Highest Risk Age Group per Diagnosis (RANK)")


  Q4 — Highest Risk Age Group per Diagnosis (RANK)
diag_1_category      age  patient_count  readmission_rate  avg_stay  avg_utilization  risk_rank
      Digestive  [20-30)            131             18.32      4.09             8.11          1
         Injury [90-100)            268             15.67      4.55             1.63          1
          Other  [20-30)           1232             15.50      3.46             5.88          1
    Circulatory  [30-40)            344             13.66      3.87             3.41          1
  Genitourinary  [70-80)           1356             12.09      4.31             2.78          1
Musculoskeletal  [70-80)           1436             11.42      4.05             1.50          1
    Respiratory  [80-90)           2645             11.27      4.89             2.60          1
      Neoplasms  [70-80)           1023             11.05      5.66             2.04          1
       Diabetes  [40-50)             58              8.62      2.60             1.03

In [8]:
q5 = """
WITH admission_stats AS (
    SELECT
        admission_type_id,
        COUNT(*)                                  AS total_patients,
        SUM(readmitted_binary)                    AS readmissions,
        ROUND(AVG(readmitted_binary)*100, 2)      AS readmission_rate,
        ROUND(AVG(time_in_hospital), 2)           AS avg_stay,
        ROUND(AVG(num_lab_procedures), 2)         AS avg_labs,
        ROUND(AVG(num_medications), 2)            AS avg_meds
    FROM patients
    GROUP BY admission_type_id
)
SELECT
    admission_type_id,
    total_patients,
    readmissions,
    readmission_rate,
    avg_stay,
    avg_labs,
    avg_meds,
    LAG(readmission_rate) OVER (
        ORDER BY readmission_rate DESC
    )                                             AS prev_rate,
    ROUND(readmission_rate - LAG(readmission_rate)
        OVER (ORDER BY readmission_rate DESC), 2) AS rate_diff
FROM admission_stats
ORDER BY readmission_rate DESC;
"""
df_q5 = run_query(q5, "Q5 — Readmission by Admission Type (LAG)")


  Q5 — Readmission by Admission Type (LAG)
 admission_type_id  total_patients  readmissions  readmission_rate  avg_stay  avg_labs  avg_meds  prev_rate  rate_diff
                 1           53979        6217.0             11.52      4.38     47.25     15.39        NaN        NaN
                 2           18478        2066.0             11.18      4.61     40.86     15.08      11.52      -0.34
                 6            5290         586.0             11.08      4.58     52.20     16.48      11.18      -0.10
                 3           18862        1960.0             10.39      4.32     34.97     18.64      11.08      -0.69
                 5            4785         495.0             10.34      3.95     26.77     15.95      10.39      -0.05
                 4              10           1.0             10.00      3.20     46.20     11.60      10.34      -0.34
                 8             320          27.0              8.44      3.06     44.43     17.48      10.00      -1.56
    

In [9]:
# Save all results for Power BI
df_q1.to_csv('../exports/risk_stratification.csv', index=False)
df_q2.to_csv('../exports/department_analysis.csv', index=False)
df_q3.to_csv('../exports/medication_impact.csv', index=False)
df_q4.to_csv('../exports/high_risk_segments.csv', index=False)
df_q5.to_csv('../exports/admission_type.csv', index=False)

# Save SQL queries
queries = """
-- =====================================================
-- MedIQ — Hospital Analytics SQL Queries
-- =====================================================

-- Q1: Patient Risk Stratification
SELECT 
    CASE 
        WHEN number_inpatient >= 5 THEN 'Very High Risk'
        WHEN number_inpatient >= 3 THEN 'High Risk'
        WHEN number_inpatient >= 1 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_tier,
    COUNT(*) AS total_patients,
    SUM(readmitted_binary) AS readmissions,
    ROUND(AVG(readmitted_binary) * 100, 2) AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2) AS avg_days_in_hospital,
    ROUND(AVG(num_medications), 2) AS avg_medications
FROM patients
GROUP BY risk_tier
ORDER BY readmission_rate_pct DESC;

-- Q2: Department Analysis
SELECT medical_specialty,
    COUNT(*) AS total_patients,
    ROUND(AVG(readmitted_binary) * 100, 2) AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2) AS avg_stay_days
FROM patients
WHERE medical_specialty != 'Unknown'
GROUP BY medical_specialty
HAVING total_patients >= 100
ORDER BY readmission_rate_pct DESC
LIMIT 15;

-- Q3: Medication Change Impact (CTE)
WITH medication_profile AS (
    SELECT 
        CASE 
            WHEN num_meds_changed = 0 THEN 'No Change'
            WHEN num_meds_changed = 1 THEN '1 Med Changed'
            WHEN num_meds_changed = 2 THEN '2 Meds Changed'
            ELSE '3+ Meds Changed'
        END AS med_change_group,
        readmitted_binary, time_in_hospital, num_medications
    FROM patients
)
SELECT med_change_group, COUNT(*) AS total_patients,
    ROUND(AVG(readmitted_binary) * 100, 2) AS readmission_rate_pct
FROM medication_profile
GROUP BY med_change_group
ORDER BY readmission_rate_pct DESC;

-- Q4: Highest Risk Age Group per Diagnosis (RANK)
WITH patient_segments AS (
    SELECT diag_1_category, age,
        ROUND(AVG(readmitted_binary)*100,2) AS readmission_rate,
        COUNT(*) AS patient_count
    FROM patients
    GROUP BY diag_1_category, age
    HAVING patient_count >= 50
),
ranked AS (
    SELECT *, RANK() OVER (
        PARTITION BY diag_1_category
        ORDER BY readmission_rate DESC
    ) AS risk_rank FROM patient_segments
)
SELECT * FROM ranked WHERE risk_rank = 1
ORDER BY readmission_rate DESC;

-- Q5: Readmission by Admission Type (LAG)
WITH admission_stats AS (
    SELECT admission_type_id,
        COUNT(*) AS total_patients,
        ROUND(AVG(readmitted_binary)*100,2) AS readmission_rate
    FROM patients GROUP BY admission_type_id
)
SELECT *, LAG(readmission_rate) OVER (ORDER BY readmission_rate DESC) AS prev_rate,
    ROUND(readmission_rate - LAG(readmission_rate)
        OVER (ORDER BY readmission_rate DESC),2) AS rate_diff
FROM admission_stats
ORDER BY readmission_rate DESC;
"""

import os
os.makedirs('../sql', exist_ok=True)
with open('../sql/queries.sql', 'w') as f:
    f.write(queries)

print("✅ All exports saved")
print(f"\n📁 exports/ folder:")
for f in sorted(os.listdir('../exports')):
    rows = len(pd.read_csv(f'../exports/{f}'))
    print(f"   {f:<35} {rows:>4} rows")
print(f"\n📁 sql/queries.sql saved")

✅ All exports saved

📁 exports/ folder:
   admission_type.csv                     8 rows
   department_analysis.csv               15 rows
   high_risk_segments.csv                 9 rows
   medication_impact.csv                  4 rows
   risk_stratification.csv                4 rows

📁 sql/queries.sql saved
